In [ ]:
from retrieval.index import DistributedIndex, load_or_initialize_index, build_index
import yaml
import os
from pathlib import Path
import torch
import json
import ir_datasets
import numpy as np
import mteb

In [ ]:
def load_model_meta_yaml(file_path : str | Path) -> dict:
    with open(file_path, "r") as f:
        return yaml.safe_load(f)

In [ ]:
path = Path("/home/rjha5/603-nvme2/arena/model_meta.yml")

In [ ]:
model_meta = load_model_meta_yaml(path)


In [ ]:
[model for model in model_meta["model_meta"].keys() if model_meta["model_meta"][model].get("size", 7000) < 2000]

In [ ]:
from models import ModelManager


model_manager = ModelManager(model_meta=model_meta)

In [ ]:
model = model_manager.load_model("intfloat/multilingual-e5-small")

In [ ]:
model_manager.load_local_index(model_name="intfloat/multilingual-e5-small", corpus="wikipedia", embedbs=1024)

# FOOBAR

In [ ]:
# write dummy passages to a jsonl file
with open("dummy_passages.jsonl", "w") as f:
    for i in range(100):
        f.write(json.dumps({"_id": i, "title": f"Dummy Passage {i}", "text": f"This is a dummy passage {i}"}) + "\n")

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
# model_name = 'intfloat/multilingual-e5-small'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)


In [ ]:
index, passages = load_or_initialize_index(dim=384, passages=["dummy_passages.jsonl"])

In [ ]:
build_index(model.bfloat16(), index, [p["text"] for p in passages], gpu_embedder_batch_size=256)

In [ ]:
index.embeddings

In [ ]:
emb_normed = torch.nn.functional.normalize(index.embeddings, p=2, dim=1)
emb_normed.norm(dim=1)

# Model Manager

In [ ]:
model_meta

In [ ]:
model_name = "nomic-ai/nomic-embed-text-v1.5"
model_name = "BAAI/bge-large-en-v1.5"
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)

In [ ]:
nomic = mteb.get_model("nomic-ai/nomic-embed-text-v1.5")
nomic

In [ ]:
hasattr(nomic, "encode_corpus")

In [ ]:
x = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=False)
type(x), x

In [ ]:
y = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=True)
type(y), y

In [ ]:
z = nomic.encode_corpus(["foobar", "baz"])
type(z), z

In [ ]:
model_meta["model_meta"][model_name]


In [ ]:
from mteb import Encoder
from typing import Any

from retrieval.index import DTYPE_TO_TORCH_DTYPE

def index_collection(model : Encoder, collection : list[str], model_meta : dict[str, Any] = {}, batch_size=32) -> DistributedIndex:
    
    index = DistributedIndex(dtype=DTYPE_TO_TORCH_DTYPE[model_meta.get("index_dtype", "float32")])
    index.init_embeddings(collection, dim=model_meta["dim"])

    print(index.embeddings.dtype)

    build_index(model, index, collection, gpu_embedder_batch_size=batch_size)

    return index

In [ ]:
index = index_collection(model, collection=[f"foobar the {i}th was a mighty king" for i in range(1000)], model_meta=model_meta["model_meta"][model_name], batch_size=64)

In [ ]:
index.search_knn(model.encode(["foobar doc 1", "foobar doc 42"], convert_to_tensor=True), topk=5)

In [ ]:
import datasets
from tqdm import tqdm

In [ ]:
wikipedia = datasets.load_dataset("orionweller/wikipedia-2024-06-24-docs", split="train")

In [ ]:
wikipedia

In [ ]:
title_text = [f"{title}\n{text}" for title, text in tqdm(zip(wikipedia["title"], wikipedia["text"]), total=len(wikipedia))]

In [ ]:
index = index_collection()

In [ ]:
from retrieval.common import load_passages_from_hf

In [ ]:
wiki = load_passages_from_hf("wikipedia", limit=None)

In [ ]:
wiki

# Simple indexing

In [ ]:
from mteb import get_model
from datasets import load_dataset
import numpy as np
from tqdm.auto import tqdm
import time
import torch

In [ ]:
wiki = load_dataset("mteb/arena-wikipedia-7-15-24", split="train")

In [ ]:
# model = get_model("jinaai/jina-embeddings-v2-base-en", revision="31b72fbf354fea65264ec54edf0b189d94b92d39")
model = get_model("BAAI/bge-large-en-v1.5", revision="d4aa6901d3a41ba39fb536a557fa166f842b0e09")
# model = get_model("mixedbread-ai/mxbai-embed-large-v1", revision="990580e27d329c7408b3741ecff85876e128e203")
# model = get_model("nomic-ai/nomic-embed-text-v1.5", revision="b0753ae76394dd36bcfb912a46018088bca48be0")

In [ ]:
start = time.time()
x = model.encode(wiki["text"][:5_000], convert_to_tensor=True, batch_size=1600, show_progress_bar=True)
end = time.time()
print(f"Time taken: {end - start} seconds")

# mem usage:

In [ ]:
batch, dim = 1024, 32

x , y = torch.zeros(dim, batch), torch.zeros(dim, batch)
z = torch.cat([x, y], dim=1)
z.shape

In [ ]:
if not isinstance(x, torch.Tensor):
    x = torch.tensor(x)

In [ ]:
gp_per_x = (x.nelement() * x.element_size() / 1024 ** 3)

gb_per_4M = (4_000_000 / x.shape[0]) * gp_per_x

hrs_per_x = (end - start) / 3600

hrs_per_4M = hrs_per_x * (4_000_000 / x.shape[0])

print(f"{gb_per_4M=:.2f}GB, {hrs_per_4M=:.2f}hrs to embed 4M docs")

# BGE large: 15 GB, 21.1 hrs, ~50% vram usage (bs=1024)
# MBAI large: 15 GB, 21 hrs, ~50% vram usage (bs=1024)
# Nomic embed v1.5: 12 GB, 10.1 hrs, ~80% vram usage (bs=1024)

In [ ]:
x = torch.load("/home/hltcoe/rjha/arena/index_wikipedia_BAAI_bge-large-en-v1.5/embeddings.0.pt")

In [ ]:
x.shape

# More Index Testing

In [ ]:
from retrieval.index import DistributedIndex
from mteb import get_model
from datasets import load_dataset
import glob
import os
import torch
import pickle
from tqdm.auto import tqdm
import pandas as pd
from fuzzywuzzy import fuzz

In [ ]:
index_dirs = glob.glob("index_wikipedia_*")
index_dirs

In [ ]:
# load 2 indices
indices = {
    index_dir : DistributedIndex()
    for index_dir in tqdm(index_dirs[-2:], desc="Creating indices")
}
for index_dir, index in tqdm(indices.items(), desc="Loading indices"):
    index.load_index(index_dir)

In [ ]:
indices["index_wikipedia_nomic-ai_nomic-embed-text-v1.5"].embeddings

In [ ]:
models = {}
for index_dir in indices:
    model_name = '/'.join(index_dir.split('_')[2:])
    print(f"{index_dir} -> {model_name}")
    models[index_dir] = get_model(model_name)

In [ ]:
qs = torch.tensor(models["index_wikipedia_nomic-ai_nomic-embed-text-v1.5"].encode_queries(["foobar doc 1", "foobar doc 42"], convert_to_tensor=True, batch_size=32, show_progress_bar=True))
indices["index_wikipedia_nomic-ai_nomic-embed-text-v1.5"].search_knn(qs, topk=5)


In [ ]:
arena_data = load_dataset("mteb/arena-results","retrieval_battle", split="data")
arena_data

In [ ]:
queries = arena_data["0_prompt"][:20]
queries


In [ ]:
docs, scores, idxs = indices["index_wikipedia_nomic-ai_nomic-embed-text-v1.5"].search_knn(torch.tensor(models["index_wikipedia_nomic-ai_nomic-embed-text-v1.5"].encode_queries(queries, convert_to_tensor=True, batch_size=32, show_progress_bar=True)), topk=40)

In [ ]:
results = [
    {
        "query" : queries[i],
        "docs" : docs[i],
        "scores" : scores[i],
        "indices" : idxs[i]
    }
    for i in range(len(queries))
]

In [ ]:
results[6]

In [ ]:
rteb_df = pd.read_csv("rteb_ranking.csv", index_col=0)
mteb_df = pd.read_csv("mteb_ranking.csv", index_col=0).sort_values("Retrieval", ascending=False)
for df in [rteb_df, mteb_df]:
    # Model column is currently a markdown link, so we need to extract the text into a new column called model_name
    df["model_name"] = df["Model"].str.extract(r"\[(.*)\]", expand=False)


In [ ]:
rteb_df.head()

In [ ]:
mteb_df.head()

In [ ]:
# check exact matches:
for model_name in model_names.keys():
    model_name = model_name.split("/")[-1]
    if model_name in ranking["model_name"].values:
        print(f"{model_name} is in the ranking")
    else:
        print(f"{model_name} is not in the ranking")

model_names.update({"foobar/bm25s": "foobar"})
model_names

In [ ]:
# get the rows that exactly match the model names
rteb_filtered_df = rteb_df[rteb_df["model_name"].isin([model_name.split("/")[-1] for model_name in model_names.keys()])]
mteb_filtered_df = mteb_df[mteb_df["model_name"].isin([model_name.split("/")[-1] for model_name in model_names.keys()])]


In [ ]:
rteb_filtered_df.head(20)

In [ ]:
mteb_filtered_df.head(20)

In [ ]:
arena_data["0_corpus"]

In [ ]:
arena_data[3389]

In [ ]:
print(max(((i, len(text), text) for i, text in enumerate(arena_data["0_prompt"])), key=lambda x: x[1]))

In [ ]:
jina = get_model("jinaai/jina-embeddings-v2-base-en", revision="31b72fbf354fea65264ec54edf0b189d94b92d39")

In [ ]:
x = jina.encode_query(["foobar doc 1" * 100] * 2444, convert_to_tensor=True, batch_size=32, show_progress_bar=True)

In [ ]:
results_dir = "arena_retrieval"
# load results into dataframes, add retrieval_model column
results_files = glob.glob(os.path.join(results_dir, "*.jsonl"))
results_dfs = []
for file in results_files:
    model_name = file.split(".")[1].replace("_", "/")
    print(f"Loading {model_name} from {file}")
    df = pd.read_json(file, lines=True)
    df["retrieval_model"] = model_name
    results_dfs.append(df)
    print(f"Loaded {len(df)} results for {model_name}")
results_dfs[0].head()


In [ ]:
results_df = pd.concat(results_dfs)
results_df.head()
len(results_df)

In [ ]:
arena_df = arena_data.to_pandas()
arena_df = arena_df[arena_df["0_corpus"] == "wikipedia"]
len(arena_df)

In [ ]:
# left join results_dfs onto arena_df
merged_df = arena_df.merge(results_df, left_on="tstamp", right_on="tstamp", how="outer")
# filter merged_df to only include rows where one of 0_model_name or 1_model_name is the row's retrieval_model
merged_df = merged_df[merged_df["0_model_name"].isin(merged_df["retrieval_model"]) | merged_df["1_model_name"].isin(merged_df["retrieval_model"])]
len(merged_df)


In [ ]:
merged_df.groupby("tstamp").head()

# Gemini Embeddings API Test

In [24]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

load_dotenv()

True

In [20]:
wiki = load_dataset("mteb/arena-wikipedia-7-15-24", split="train")

In [21]:
docs = [f"{title}\n\n{text}" for title, text in tqdm(zip(wiki["title"], wiki["text"]), total=len(wiki))]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3811232/3811232 [02:14<00:00, 28400.46it/s]


In [3]:
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [11]:
[m for m in client.models.list() if "embed" in m.name]

[Model(
   description='Obtain a distributed representation of a text.',
   display_name='Embedding Gecko',
   input_token_limit=1024,
   name='models/embedding-gecko-001',
   output_token_limit=1,
   supported_actions=[
     'embedText',
     'countTextTokens',
   ],
   tuned_model_info=TunedModelInfo(),
   version='001'
 ),
 Model(
   description='Obtain a distributed representation of a text.',
   display_name='Embedding 001',
   input_token_limit=2048,
   name='models/embedding-001',
   output_token_limit=1,
   supported_actions=[
     'embedContent',
   ],
   tuned_model_info=TunedModelInfo(),
   version='001'
 ),
 Model(
   description='Obtain a distributed representation of a text.',
   display_name='Text Embedding 004',
   input_token_limit=2048,
   name='models/text-embedding-004',
   output_token_limit=1,
   supported_actions=[
     'embedContent',
   ],
   tuned_model_info=TunedModelInfo(),
   version='004'
 ),
 Model(
   description='Obtain a distributed representation of a

In [30]:
lengths = [len(d) for d in docs]

In [ ]:
sum(lengths)

1862921966

In [ ]:
results = client.models.embed_content(
    model="gemini-embedding-001",
    contents=docs[:100],
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"), 
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}